In [ ]:
from pathlib import Path
import re
import string
from collections import Counter
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
DATASET_ID = 331
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)


## 1. Load Dataset

The notebook first tries to load the UCI Sentiment Labelled Sentences dataset automatically using `ucimlrepo` with dataset id `331`.

If that fails, it falls back to local files such as:

- `amazon_cells_labelled.txt`
- `imdb_labelled.txt`
- `yelp_labelled.txt`
- or a combined local CSV/TSV file if available


In [ ]:
def normalize_uci_dataframe(features_df: pd.DataFrame, targets_df: pd.DataFrame) -> pd.DataFrame:
    df = pd.concat([features_df.reset_index(drop=True), targets_df.reset_index(drop=True)], axis=1)

    text_candidates = ["sentence", "text", "review", "content"]
    label_candidates = ["label", "sentiment", "class", "target"]
    source_candidates = ["source", "origin", "dataset"]

    def first_existing(candidates):
        for candidate in candidates:
            if candidate in df.columns:
                return candidate
        return None

    text_col = first_existing(text_candidates) or df.columns[0]
    label_col = first_existing(label_candidates) or df.columns[-1]
    source_col = first_existing(source_candidates)

    normalized = pd.DataFrame(
        {
            "text": df[text_col].astype(str),
            "label": pd.to_numeric(df[label_col], errors="coerce").astype("Int64"),
        }
    )

    if source_col is not None:
        normalized["source"] = df[source_col].astype(str)
    else:
        normalized["source"] = "uci"

    return normalized[["text", "label", "source"]]


def load_from_ucimlrepo():
    from ucimlrepo import fetch_ucirepo

    dataset = fetch_ucirepo(id=DATASET_ID)
    features_df = dataset.data.features.copy()
    targets_df = dataset.data.targets.copy()
    return normalize_uci_dataframe(features_df, targets_df)


def load_from_local_files():
    dataset_parts = []
    candidate_directories = [
        Path("."),
        Path("data"),
        Path("data/sentiment_labelled_sentences"),
        Path("data/sentiment labelled sentences"),
        Path("sentiment_labelled_sentences"),
        Path("sentiment labelled sentences"),
    ]

    txt_mapping = {
        "amazon": "amazon_cells_labelled.txt",
        "imdb": "imdb_labelled.txt",
        "yelp": "yelp_labelled.txt",
    }

    for source_name, filename in txt_mapping.items():
        matched_path = None
        for directory in candidate_directories:
            candidate_path = directory / filename
            if candidate_path.exists():
                matched_path = candidate_path
                break

        if matched_path is not None:
            part_df = pd.read_csv(
                matched_path,
                sep="\t",
                header=None,
                names=["text", "label"],
            )
            part_df["source"] = source_name
            dataset_parts.append(part_df)

    if dataset_parts:
        return pd.concat(dataset_parts, ignore_index=True)

    csv_candidates = [
        Path("sentiment_labelled_sentences.csv"),
        Path("sentiment_labelled_sentences.tsv"),
        Path("data/sentiment_labelled_sentences.csv"),
        Path("data/sentiment_labelled_sentences.tsv"),
    ]

    for csv_path in csv_candidates:
        if csv_path.exists():
            sep = "\t" if csv_path.suffix.lower() == ".tsv" else ","
            local_df = pd.read_csv(csv_path, sep=sep)
            local_df.columns = [col.strip().lower() for col in local_df.columns]

            text_col = "text" if "text" in local_df.columns else "sentence"
            label_col = "label" if "label" in local_df.columns else "sentiment"

            normalized = pd.DataFrame(
                {
                    "text": local_df[text_col].astype(str),
                    "label": pd.to_numeric(local_df[label_col], errors="coerce").astype("Int64"),
                    "source": local_df["source"].astype(str) if "source" in local_df.columns else "local_csv",
                }
            )
            return normalized

    raise FileNotFoundError(
        "Local fallback dataset not found. Add the original TXT files or a local CSV/TSV copy."
    )


def load_sentiment_dataset():
    loader_used = None
    ucimlrepo_error = None

    try:
        df_loaded = load_from_ucimlrepo()
        loader_used = "ucimlrepo"
    except Exception as exc:
        ucimlrepo_error = str(exc)
        df_loaded = load_from_local_files()
        loader_used = "local_files"

    return df_loaded, loader_used, ucimlrepo_error


df, loader_used, ucimlrepo_error = load_sentiment_dataset()
df["label"] = pd.to_numeric(df["label"], errors="coerce").astype(int)

print(f"Loader used: {loader_used}")
if ucimlrepo_error:
    print(f"ucimlrepo error (fallback activated): {ucimlrepo_error}")

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
display(df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count"))


## 2. Exploratory Data Analysis

We inspect:

- missing values,
- duplicates,
- class balance,
- text length distribution,
- most frequent words in positive and negative reviews.


In [ ]:
df["text_length_chars"] = df["text"].astype(str).str.len()
df["text_length_words"] = df["text"].astype(str).str.split().str.len()

summary_df = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_values": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    }
).sort_values(["missing_values", "unique_values"], ascending=[False, False])

duplicate_rows = df.duplicated().sum()

display(summary_df)
print(f"Duplicate rows: {duplicate_rows}")


In [ ]:
target_distribution = (
    df["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Negative", 1: "Positive"})
    .rename_axis("sentiment")
    .reset_index(name="count")
)
target_distribution["pct"] = (target_distribution["count"] / len(df) * 100).round(2)
display(target_distribution)

plt.figure(figsize=(7, 4.5))
ax = sns.countplot(data=df, x="label", palette="Set2")
ax.set_xticklabels(["Negative", "Positive"])
ax.set_title("Target Distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="text_length_chars", hue="label", bins=30, kde=True, ax=axes[0], palette="viridis")
axes[0].set_title("Text Length Distribution (Characters)")
axes[0].set_xlabel("Characters")
axes[0].legend(["Negative", "Positive"])

sns.histplot(data=df, x="text_length_words", hue="label", bins=30, kde=True, ax=axes[1], palette="magma")
axes[1].set_title("Text Length Distribution (Words)")
axes[1].set_xlabel("Words")
axes[1].legend(["Negative", "Positive"])

plt.tight_layout()
plt.show()


## 3. Text Preprocessing

Preprocessing remains intentionally simple and readable:

- lowercase the text,
- remove punctuation,
- remove extra spaces,
- remove stopwords to reduce noise in very short review sentences.

This keeps the pipeline lightweight while still improving TF-IDF quality.


In [ ]:
STOPWORDS = set(ENGLISH_STOP_WORDS)


def clean_text(text: str, remove_stopwords: bool = True) -> str:
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    if remove_stopwords:
        tokens = [token for token in text.split() if token not in STOPWORDS]
        text = " ".join(tokens)

    return text


df["text_clean"] = df["text"].astype(str).apply(clean_text)

preview_df = df[["text", "text_clean", "label"]].head(10)
display(preview_df)


In [ ]:
def top_words_by_label(frame: pd.DataFrame, label_value: int, top_n: int = 15):
    tokens = " ".join(frame.loc[frame["label"] == label_value, "text_clean"]).split()
    counter = Counter(tokens)
    top_items = counter.most_common(top_n)
    return pd.DataFrame(top_items, columns=["word", "count"])


negative_top_words = top_words_by_label(df, 0, top_n=15)
positive_top_words = top_words_by_label(df, 1, top_n=15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=negative_top_words, x="count", y="word", ax=axes[0], color="#e76f51")
axes[0].set_title("Most Frequent Words - Negative Reviews")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Word")

sns.barplot(data=positive_top_words, x="count", y="word", ax=axes[1], color="#2a9d8f")
axes[1].set_title("Most Frequent Words - Positive Reviews")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Word")

plt.tight_layout()
plt.show()


## 4. Train-Test Split

We keep the class distribution stable with a **stratified split**.


In [ ]:
X = df["text_clean"].copy()
y = df["label"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_balance_df = pd.DataFrame(
    {
        "train_count": y_train.value_counts().sort_index(),
        "test_count": y_test.value_counts().sort_index(),
        "train_pct": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
        "test_pct": (y_test.value_counts(normalize=True).sort_index() * 100).round(2),
    }
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
display(split_balance_df)


## 5. Feature Extraction with TF-IDF

We compare:

- **Unigrams**: single words only
- **Unigrams + Bigrams**: single words and two-word expressions

The comparison is done on the training set only to avoid data leakage.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
ngram_options = {
    "Unigrams": (1, 1),
    "Unigrams + Bigrams": (1, 2),
}

ngram_rows = []
for label_name, ngram_range in ngram_options.items():
    comparison_pipeline = Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(ngram_range=ngram_range, min_df=2, max_df=0.95)),
            ("model", LogisticRegression(max_iter=2000, solver="liblinear")),
        ]
    )

    scores = cross_val_score(
        comparison_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=None,
    )

    ngram_rows.append(
        {
            "Representation": label_name,
            "ngram_range": ngram_range,
            "CV F1 Mean": scores.mean(),
            "CV F1 Std": scores.std(),
        }
    )

ngram_comparison_df = pd.DataFrame(ngram_rows).sort_values("CV F1 Mean", ascending=False).reset_index(drop=True)
best_ngram_range = ngram_comparison_df.loc[0, "ngram_range"]
best_ngram_label = ngram_comparison_df.loc[0, "Representation"]

display(ngram_comparison_df.style.format({"CV F1 Mean": "{:.4f}", "CV F1 Std": "{:.4f}"}))
print(f"Selected TF-IDF configuration: {best_ngram_label} with ngram_range={best_ngram_range}")


## 6. Model Training

We compare at least four classical text classifiers:

- Logistic Regression
- Linear SVM
- SGDClassifier
- Multinomial Naive Bayes


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, solver="liblinear"),
    "Linear SVM": LinearSVC(),
    "SGDClassifier": SGDClassifier(loss="hinge", random_state=RANDOM_STATE),
    "Multinomial Naive Bayes": MultinomialNB(),
}


def build_text_pipeline(model, ngram_range):
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(ngram_range=ngram_range, min_df=2, max_df=0.95)),
            ("model", model),
        ]
    )


## 7. Model Evaluation

For each model we compute:

- Accuracy
- Precision
- Recall
- F1-score

We also generate:

- classification reports,
- confusion matrices,
- a comparison table,
- a bar chart of performance.


In [ ]:
evaluation_rows = []
classification_reports = {}
fitted_pipelines = {}
test_predictions = {}

for model_name, estimator in models.items():
    pipeline = build_text_pipeline(estimator, best_ngram_range)
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)

    fitted_pipelines[model_name] = pipeline
    test_predictions[model_name] = preds
    classification_reports[model_name] = classification_report(
        y_test,
        preds,
        target_names=["Negative", "Positive"],
        zero_division=0,
    )

    evaluation_rows.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds, pos_label=1, zero_division=0),
            "Recall": recall_score(y_test, preds, pos_label=1, zero_division=0),
            "F1": f1_score(y_test, preds, pos_label=1, zero_division=0),
        }
    )

comparison_df = pd.DataFrame(evaluation_rows).sort_values(
    by=["F1", "Accuracy", "Precision", "Recall"],
    ascending=False,
).reset_index(drop=True)

display(comparison_df.style.format({"Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"}))


In [ ]:
plot_df = comparison_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score",
)

plt.figure(figsize=(11, 5.5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", palette="viridis")
plt.title("Model Performance Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
n_models = len(test_predictions)
n_cols = 2
n_rows = int(np.ceil(n_models / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, (model_name, preds) in zip(axes, test_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        preds,
        display_labels=["Negative", "Positive"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(model_name)

for ax in axes[n_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
for model_name in comparison_df["Model"]:
    print("=" * 100)
    print(model_name)
    print("=" * 100)
    print(classification_reports[model_name])


## 8. Model Selection

The best model is selected by **F1-score**, because F1 balances precision and recall and is especially useful when we want a reliable positive/negative sentiment decision rather than only raw accuracy.


In [ ]:
best_model_name = comparison_df.loc[0, "Model"]
best_pipeline = fitted_pipelines[best_model_name]
best_f1 = comparison_df.loc[0, "F1"]

print(f"Best model: {best_model_name}")
print(f"Best F1-score: {best_f1:.4f}")
print(
    "Why selected: it achieved the highest F1-score on the holdout test set "
    "while preserving strong overall classification quality."
)


## 9. Test Predictions on Business / Market Sentences

We now test the final model on custom sentences related to customer feedback and market sentiment.


In [ ]:
custom_examples = [
    "Customers love this product and the market demand is growing.",
    "Users complain about high prices and poor service.",
    "The idea is interesting but the market is uncertain.",
]

custom_predictions = best_pipeline.predict([clean_text(text) for text in custom_examples])

custom_results_df = pd.DataFrame(
    {
        "sentence": custom_examples,
        "predicted_label": custom_predictions,
    }
)
custom_results_df["predicted_sentiment"] = custom_results_df["predicted_label"].map({0: "Negative", 1: "Positive"})

if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    custom_probabilities = best_pipeline.predict_proba([clean_text(text) for text in custom_examples])[:, 1]
    custom_results_df["positive_probability"] = custom_probabilities.round(4)
elif hasattr(best_pipeline.named_steps["model"], "decision_function"):
    decision_scores = best_pipeline.decision_function([clean_text(text) for text in custom_examples])
    custom_results_df["decision_score"] = np.round(decision_scores, 4)

display(custom_results_df)


## 10. Save Final Pipeline

The best model and TF-IDF vectorizer are saved together inside a reusable scikit-learn pipeline. This makes the artifact directly reusable in FastAPI.


In [ ]:
sentiment_model_path = ARTIFACTS_DIR / "sentiment_analysis_pipeline.joblib"

joblib.dump(
    {
        "pipeline": best_pipeline,
        "best_model_name": best_model_name,
        "best_ngram_range": best_ngram_range,
        "best_ngram_label": best_ngram_label,
        "target_mapping": {0: "Negative", 1: "Positive"},
        "preprocessing_note": "lowercase + punctuation removal + extra-space cleanup + English stopword removal",
        "dataset_loader_used": loader_used,
    },
    sentiment_model_path,
)

print(f"Sentiment pipeline saved to: {sentiment_model_path.resolve()}")


## 11. Conclusion

This notebook trained and compared several classical sentiment analysis models on the UCI Sentiment Labelled Sentences dataset.

### Final Outcome

- the best model was selected by F1-score,
- TF-IDF text features were used with a leakage-safe train/test workflow,
- the final artifact was saved as a reusable pipeline for deployment.

### Integration into the AI Business Intelligence Platform

This model can be integrated into the platform to:

- classify customer and market opinions automatically,
- score public/product feedback as positive or negative,
- enrich idea-validation reports with sentiment indicators,
- support future FastAPI endpoints for real-time opinion analysis.
